# Practical 1:An introduction to viewing and manipulating medical imaging data

## 1. Viewing images with ITK-SNAP

#### This section is completed using ITK-SNAP.

## 2. Viewing and understanding the NIfTI header with NiBabel

### 2.1. Reading and displaying the NIfTI header

In [2]:
# import packages
import numpy as np
import nibabel as nib

In [ ]:
# load the nifti file saved in section 1
ct_for_pet_nii = nib.load("data/practical1/CT_for_PET.nii.gz")

#this returns a Nifti1Image object
print(type(ct_for_pet_nii))

In [ ]:
# display the size/shape of the image
print(ct_for_pet_nii.shape)

# display the data type of the image
print(ct_for_pet_nii.get_data_dtype())

# display the affine matrix for the image
print(ct_for_pet_nii.affine)

In [ ]:
# print the header
print(ct_for_pet_nii.header)

### 2.2 Specifying the affine transform

In [ ]:
# print the matrix represented by the qform
print(ct_for_pet_nii.get_qform())

## 3. Modifying NIfTI images and headers with NiBabel

### 3.1. Changing the data type and qform code

In [ ]:
# read the image data of the Nifti1Image object
# it gets loaded as a numpy array and casted to float64
ct_for_pet_img = ct_for_pet_nii.get_fdata()

# check the type of the array
print(type(ct_for_pet_img))

# check the type of the array elements
print(ct_for_pet_img.dtype)

In [ ]:
# change the data type to float32
ct_for_pet_nii.set_data_dtype(np.float32)

# set the qform code to unkown (0)
ct_for_pet_nii.set_qform(None, code='unknown')

# check the header has been updated
print(ct_for_pet_nii.header)

# save the float32 image to disk
nib.save(ct_for_pet_nii, 'data/practical1/CT_for_PET_float32.nii.gz')

#### Check data types with NiBabel

In [ ]:
# load the new image and check the data type is float32
ct_for_pet_float32_nii = nib.load("data/practical1/CT_for_PET_float32.nii.gz")
print(ct_for_pet_float32_nii.get_data_dtype())

# load the original image and check the data type is still int16
ct_for_pet_orig_nii = nib.load("data/practical1/CT_for_PET.nii.gz")
print(ct_for_pet_orig_nii.get_data_dtype())

### 3.2. Cropping images

In [ ]:
# create a new array containing a copy of the desired slices
x_first = 91
x_last = 390
y_first = 131
y_last = 375
z_first = 21
z_last = 155
ct_for_pet_cropped_img = ct_for_pet_img[x_first:x_last+1, y_first:y_last+1, z_first:z_last+1].copy()

# check the size of the arrays containing the original image and the cropped image
print(ct_for_pet_img.shape)
print(ct_for_pet_cropped_img.shape)

In [ ]:
# create a new Nifti1Image object using the header and affine from the uncropped image
ct_for_pet_cropped_nii = nib.nifti1.Nifti1Image(ct_for_pet_cropped_img, ct_for_pet_nii.affine, ct_for_pet_nii.header)

# check the shape and header of the new Nifti1Image object
print(ct_for_pet_cropped_nii.shape)
print(ct_for_pet_cropped_nii.header)

# save the cropped image
nib.save(ct_for_pet_cropped_nii, "data/practical1/CT_for_PET_cropped.nii.gz")

In [ ]:
# calculate the world coordinates of first voxel in the cropped image
cropped_origin = ct_for_pet_nii.affine @ np.array([x_first, y_first, z_first, 1])
print(cropped_origin)

# use these to update the corresponding values in the affine matrix for the cropped image
ct_for_pet_cropped_nii.affine[0,3] = cropped_origin[0]
ct_for_pet_cropped_nii.affine[1,3] = cropped_origin[1]
ct_for_pet_cropped_nii.affine[2,3] = cropped_origin[2]
print(ct_for_pet_cropped_nii.affine)

# save the cropped image - this also updates the sform values in the header using the updated affine
print(ct_for_pet_cropped_nii.header.get_sform())
nib.save(ct_for_pet_cropped_nii, "data/practical1/CT_for_PET_cropped_aligned.nii.gz")
print(ct_for_pet_cropped_nii.header.get_sform())


In [ ]:
# we can do the same as above using nibabel's slicer attribute
ct_for_pet_slicer_nii = ct_for_pet_nii.slicer[x_first:x_last+1, y_first:y_last+1, z_first:z_last+1]
print(ct_for_pet_slicer_nii.header)

### 3.3. Aligning images

In [61]:
# TODO: Add you code here to align the centre of the inhale_BH_CT with the centre of CT_for_PET

# find the centre of the ct_for_pet image in voxel coordinates
ct_for_pet_centre_vox = np.ones(4)
ct_for_pet_centre_vox[0] = (ct_for_pet_nii.shape[0] - 1) / 2
ct_for_pet_centre_vox[1] = (ct_for_pet_nii.shape[1] - 1) / 2
ct_for_pet_centre_vox[2] = (ct_for_pet_nii.shape[2] - 1) / 2

# find the centre of the ct_for_pet image in world coordinates
ct_for_pet_centre_world = ct_for_pet_nii.affine @ ct_for_pet_centre_vox

# find the centre of the inhale_ct image in voxel coordinates
inhale_ct_nii = nib.load('data/practical1/inhale_BH_CT.nii.gz')
inhale_ct_centre_vox = np.ones(4)
inhale_ct_centre_vox[0] = (inhale_ct_nii.shape[0] - 1) / 2
inhale_ct_centre_vox[1] = (inhale_ct_nii.shape[1] - 1) / 2
inhale_ct_centre_vox[2] = (inhale_ct_nii.shape[2] - 1) / 2

# find the centre of the inhale_ct image in world coordinates
inhale_ct_centre_world = inhale_ct_nii.affine @ inhale_ct_centre_vox

# find the translation between the centres of the images (in world coordinates)
translation = ct_for_pet_centre_world - inhale_ct_centre_world

# add the translation to the cooresponding elements of the affine matrix for the inhale_ct image
inhale_ct_nii.affine[0,3] = inhale_ct_nii.affine[0,3] + translation[0]
inhale_ct_nii.affine[1,3] = inhale_ct_nii.affine[1,3] + translation[1]
inhale_ct_nii.affine[2,3] = inhale_ct_nii.affine[2,3] + translation[2]

# save the aligned image - the updated affine will be used to update the header when it is saved
nib.save(inhale_ct_nii, "data/practical1/inhale_BH_CT_aligned.nii.gz")